In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
class ICA:
    def __init__(self, nsource=2, nspeaker=1024):
        self.nsource = nsource
        self.nspeaker = nspeaker
        self.W = np.random.randn(nsource, nspeaker)

    def whiten(self, X):
        X = X - X.mean(axis=1, keepdims=True)

        cov = np.cov(X)
        d, E = np.linalg.eigh(cov)

        D_inv = np.diag(1.0 / np.sqrt(d + 1e-8))
        X_white = E @ D_inv @ E.T @ X
        return X_white

    def train(self, X, lr=1e-3, epochs=20):
        # X: (examples, frames, features)
        X = X.reshape(-1, self.nspeaker).T  # (features, samples)
        X = self.whiten(X)

        for _ in range(epochs):
            Y = self.W @ X

            # nonlinearity (tanh is standard ICA choice)
            g = np.tanh(Y)
            g_der = 1 - g**2

            dW = (np.eye(self.nsource) + (g @ Y.T) / X.shape[1]) @ self.W

            self.W += lr * dW

    def predict(self, X):
        # (n_frames, nspeaker)
        S = self.W @ X.T   # (nsource, n_frames)

        return S.T   # (n_frames, nsource)

In [ ]:
!pip install kaggle

In [ ]:
!kaggle datasets download -d freecolabgpu/audio-source-separation

Dataset URL: https://www.kaggle.com/datasets/freecolabgpu/audio-source-separation
License(s): unknown
100% 971M/971M [00:09<00:00, 106MB/s]



In [ ]:
import zipfile

with zipfile.ZipFile("audio-source-separation.zip", "r") as z:
    z.extractall("audio_source_separation")

LOADING DATASET COPIED FROM LLM AS IM TOO LAZY FOR TS


In [ ]:
import os
import numpy as np
import librosa

DATASET_DIR = "audio_source_separation/mixed_dataset/mixtures"   # adjust path

SR = 16000                # target sample rate
FRAME_LEN = 1024          # samples per frame
HOP_LEN = FRAME_LEN       # non-overlapping frames

examples = []

for fname in sorted(os.listdir(DATASET_DIR)):
    if not fname.endswith(".wav"):
        continue

    path = os.path.join(DATASET_DIR, fname)

    y, _ = librosa.load(path, sr=SR, mono=True)

    n_frames = len(y) // FRAME_LEN
    y = y[: n_frames * FRAME_LEN]

    frames = y.reshape(n_frames, FRAME_LEN)

    examples.append(frames)

min_frames = min(x.shape[0] for x in examples)

X = np.stack(
    [x[:min_frames] for x in examples],
    axis=0
)

print(X.shape)

(1000, 156, 1024)


In [ ]:
model = ICA()

model.W = omega

In [ ]:
from scipy.io import wavfile

sample_rate, audio = wavfile.read(DATASET_DIR+"/mix_0003.wav")
audio.shape

(220500,)

In [ ]:
_nsample = audio.shape[0]//1024
audio=audio[:_nsample*1024]
audio = audio.reshape((_nsample, 1024))
audio.shape

(215, 1024)

In [ ]:
omega = model.W
omega

array([[ 0.16294049,  0.0675486 , -1.66556644, ...,  0.00503792,
        -3.07832883,  0.5243845 ],
       [-1.35373382, -1.75370082, -1.81673638, ..., -0.35107834,
        -1.09846311, -1.56172061]])

In [ ]:
sources = np.array([model.predict(audio[i]) for i in range(215)])


In [ ]:
sources.shape

(215, 2)

In [ ]:
from IPython.display import Audio

Audio(sources[:][0].reshape(-1), rate=22050)